# P4-4 · Bài 7 · Agent

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q langchain-anthropic langchain-core langgraph

## Lý thuyết

In [ ]:
def run_agent(user_msg, tools, tool_fns, max_steps=8):
    messages = [{"role": "user", "content": user_msg}]
    for step in range(max_steps):
        resp = client.messages.create(model="claude-haiku-5-5", max_tokens=1024,
                                      system=SYSTEM, tools=tools, messages=messages)
        messages.append({"role": "assistant", "content": resp.content})
        calls = [b for b in resp.content if b.type == "tool_use"]
        if not calls:
            return resp.content[0].text, messages                  # xong việc
        results = []
        for c in calls:
            try:
                out = tool_fns[c.name](**c.input)
            except Exception as e:                                 # lỗi tool trả lại cho model, không làm sập vòng lặp
                out = f"Lỗi khi gọi {c.name}: {e}"
            results.append({"type": "tool_result", "tool_use_id": c.id, "content": str(out)[:4000]})
        messages.append({"role": "user", "content": results})
    return "Chưa hoàn thành sau số bước tối đa, đã chuyển cho nhân viên.", messages

In [ ]:
# Cần ANTHROPIC_API_KEY: chạy trên máy bạn hoặc Colab
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool

@tool
def get_order_status(order_id: str) -> str:
    """Tra trạng thái đơn hàng theo mã đơn, ví dụ DH1024."""
    return {"DH1024": "đang giao, dự kiến 6/11"}.get(order_id, f"Không tìm thấy {order_id}")

class State(TypedDict):
    messages: Annotated[list, add_messages]          # reducer: nối thêm message mới

llm = ChatAnthropic(model="claude-haiku-5-5").bind_tools([get_order_status])
graph = StateGraph(State)
graph.add_node("agent", lambda s: {"messages": [llm.invoke(s["messages"])]})
graph.add_node("tools", ToolNode([get_order_status]))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)     # có gọi tool thì sang "tools", không thì kết thúc
graph.add_edge("tools", "agent")
app = graph.compile()
out = app.invoke({"messages": [("user", "Đơn DH1024 của tôi tới đâu rồi?")]}, {"recursion_limit": 10})

## Bài tập 5: Agent không dùng framework {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import json, anthropic
client = anthropic.Anthropic()

def run_agent(question, tools, funcs, max_steps=8, model="claude-haiku-5-5"):
    messages = [{"role": "user", "content": question}]
    for step in range(1, max_steps + 1):
        resp = client.messages.create(model=model, max_tokens=1000, tools=tools, messages=messages)
        messages.append({"role": "assistant", "content": resp.content})
        calls = [b for b in resp.content if b.type == "tool_use"]
        if not calls:
            return "".join(b.text for b in resp.content if b.type == "text")
        results = []
        for c in calls:
            try:
                out = funcs[c.name](**c.input)
                results.append({"type": "tool_result", "tool_use_id": c.id, "content": json.dumps(out, ensure_ascii=False)})
            except Exception as e:
                results.append({"type": "tool_result", "tool_use_id": c.id, "content": f"Lỗi: {e}", "is_error": True})
            print(f"[bước {step}] {c.name}({c.input}) → {results[-1]['content'][:120]}")
        messages.append({"role": "user", "content": results})
    return "Dừng: vượt quá số bước cho phép."

## Bài tập 6: Workflow cố định cho email khách hàng {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import re
emails = ["Cho mình hỏi đơn DH1024 bao giờ giao vậy?", "Shop xuất giúp hóa đơn VAT cho đơn DH0988 nhé",
          "Bàn giao tới bị gãy chân, quá thất vọng!", "Đơn của tôi đâu rồi???", "Hóa đơn đơn DH1100 gửi về email nào ạ?"]
DON = {"DH1024": "đang giao, dự kiến thứ Sáu", "DH0988": "đã giao", "DH1100": "đã giao"}

def xu_ly(email):
    # TODO: trả về dict {"loai", "ma_don", "tra_loi", "chuyen_nguoi"}
    return {}

In [ ]:
def xu_ly(email):
    e = email.lower()
    loai = "hoa_don" if "hóa đơn" in e else "khieu_nai" if any(w in e for w in ["gãy", "thất vọng", "hỏng"]) else "don_hang"
    m = re.search(r"DH\d+", email); ma = m.group() if m else None
    if loai == "khieu_nai" or ma is None:
        return {"loai": loai, "ma_don": ma, "tra_loi": None, "chuyen_nguoi": True}
    tra_loi = (f"Đơn {ma}: {DON.get(ma, 'không tìm thấy')}." if loai == "don_hang"
               else f"Shop sẽ gửi hóa đơn đơn {ma} trong 24 giờ.")
    return {"loai": loai, "ma_don": ma, "tra_loi": tra_loi, "chuyen_nguoi": False}

for e in emails:
    print(xu_ly(e))

{'loai': 'don_hang', 'ma_don': 'DH1024', 'tra_loi': 'Đơn DH1024: đang giao, dự kiến thứ Sáu.', 'chuyen_nguoi': False}
{'loai': 'hoa_don', 'ma_don': 'DH0988', 'tra_loi': 'Shop sẽ gửi hóa đơn đơn DH0988 trong 24 giờ.', 'chuyen_nguoi': False}
{'loai': 'khieu_nai', 'ma_don': None, 'tra_loi': None, 'chuyen_nguoi': True}
{'loai': 'don_hang', 'ma_don': None, 'tra_loi': None, 'chuyen_nguoi': True}
{'loai': 'hoa_don', 'ma_don': 'DH1100', 'tra_loi': 'Shop sẽ gửi hóa đơn đơn DH1100 trong 24 giờ.', 'chuyen_nguoi': False}


## Bài tập 7: Sửa một tool khó dùng {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import json, random
random.seed(0)
ORDERS = [{"order_id": f"DH{1000 + i}", "phone": "0912345678" if i % 3 else "0987654321",
           "status": random.choice(["đang giao", "đã giao", "đã hủy"]), "total": random.randint(2, 40) * 50000,
           "items": [{"sku": f"SKU{j}", "warehouse_log": "x" * 200} for j in range(3)], "internal_note": "..." * 50}
          for i in range(30)]

def search_orders(q):
    return json.dumps([o for o in ORDERS if o["phone"] == q["phone"]])

def tim_don(sdt, trang_thai=None, gioi_han=5):
    # TODO
    return ""

In [ ]:
def tim_don(sdt, trang_thai=None, gioi_han=5):
    """Tìm đơn hàng theo số điện thoại khách. trang_thai: 'đang giao' | 'đã giao' | 'đã hủy' (tùy chọn)."""
    if not (len(sdt) == 10 and sdt.isdigit() and sdt.startswith("0")):
        return "Lỗi: sdt phải có 10 chữ số và bắt đầu bằng 0, ví dụ 0912345678."
    ds = [o for o in ORDERS if o["phone"] == sdt and (trang_thai is None or o["status"] == trang_thai)]
    gon = [{"ma": o["order_id"], "trang_thai": o["status"], "tong": o["total"]} for o in ds[:gioi_han]]
    return json.dumps({"don": gon, "con_lai": max(0, len(ds) - gioi_han)}, ensure_ascii=False)

print("tool cũ:", len(search_orders({"phone": "0912345678"})), "ký tự")
moi = tim_don("0912345678", trang_thai="đang giao")
print("tool mới:", len(moi), "ký tự →", moi)
print(tim_don("912345678"))

tool cũ: 19844 ký tự
tool mới: 330 ký tự → {"don": [{"ma": "DH1001", "trang_thai": "đang giao", "tong": 900000}, {"ma": "DH1008", "trang_thai": "đang giao", "tong": 900000}, {"ma": "DH1010", "trang_thai": "đang giao", "tong": 1050000}, {"ma": "DH1011", "trang_thai": "đang giao", "tong": 300000}, {"ma": "DH1014", "trang_thai": "đang giao", "tong": 1200000}], "con_lai": 2}
Lỗi: sdt phải có 10 chữ số và bắt đầu bằng 0, ví dụ 0912345678.


## Bài tập 8: Chặn agent đi vòng tròn {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import json

def fake_policy(lich_su):
    """Trả về ('tool', tên, input) hoặc ('xong', câu trả lời). Bị kẹt: gọi lại tra_don mãi, trừ khi đã bị nhắc."""
    if any(m.startswith("NHẮC:") for m in lich_su):
        return ("xong", "Đơn DH1024 đang giao, dự kiến thứ Sáu.")
    return ("tool", "tra_don", {"ma": "DH1024"})

def chay(policy, max_buoc=10):
    lich_su, dem = [], {}
    # TODO
    return "het_buoc", max_buoc

In [ ]:
def chay(policy, max_buoc=10):
    lich_su, dem = [], {}
    for buoc in range(1, max_buoc + 1):
        hanh_dong = policy(lich_su)
        if hanh_dong[0] == "xong":
            print("trả lời:", hanh_dong[1])
            return "xong", buoc
        _, ten, inp = hanh_dong
        khoa = (ten, json.dumps(inp, sort_keys=True))
        dem[khoa] = dem.get(khoa, 0) + 1
        lich_su.append(f"KẾT QUẢ {ten}({inp}): đang giao")
        if dem[khoa] == 3:
            lich_su.append("NHẮC: bạn đã gọi đúng tool này 3 lần với cùng đầu vào; hãy trả lời bằng thông tin đã có.")
            print(f"bước {buoc}: phát hiện lặp, chèn nhắc nhở")
    return "het_buoc", max_buoc

print(chay(fake_policy))

bước 3: phát hiện lặp, chèn nhắc nhở
trả lời: Đơn DH1024 đang giao, dự kiến thứ Sáu.
('xong', 4)


## Bài tập 9: Đồ thị trạng thái bằng LangGraph {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    tin_cay: float
    so_lan_hoi: int
    duong_di: list

def phan_loai(s):
    return {"duong_di": s["duong_di"] + ["phan_loai"]}

def hoi_lai(s):
    return {"tin_cay": s["tin_cay"] + 0.3, "so_lan_hoi": s["so_lan_hoi"] + 1, "duong_di": s["duong_di"] + ["hoi_lai"]}

def tra_loi(s):
    return {"duong_di": s["duong_di"] + ["tra_loi"]}

def chuyen_nguoi(s):
    return {"duong_di": s["duong_di"] + ["chuyen_nguoi"]}
# TODO: hàm chon(s) cho cạnh có điều kiện; build graph; app = g.compile(); chạy 2 trường hợp

In [ ]:
def chon(s):
    if s["tin_cay"] >= 0.7:
        return "tra_loi"
    return "hoi_lai" if s["so_lan_hoi"] < 2 else "chuyen_nguoi"

g = StateGraph(State)
for ten, f in [("phan_loai", phan_loai), ("hoi_lai", hoi_lai), ("tra_loi", tra_loi), ("chuyen_nguoi", chuyen_nguoi)]:
    g.add_node(ten, f)
g.add_edge(START, "phan_loai")
g.add_conditional_edges("phan_loai", chon)
g.add_edge("hoi_lai", "phan_loai")
g.add_edge("tra_loi", END); g.add_edge("chuyen_nguoi", END)
app = g.compile()

kq_02 = app.invoke({"tin_cay": 0.2, "so_lan_hoi": 0, "duong_di": []})
kq_05 = app.invoke({"tin_cay": 0.5, "so_lan_hoi": 0, "duong_di": []})
print("0,2:", " → ".join(kq_02["duong_di"]))
print("0,5:", " → ".join(kq_05["duong_di"]))

0,2: phan_loai → hoi_lai → phan_loai → hoi_lai → phan_loai → tra_loi
0,5: phan_loai → hoi_lai → phan_loai → tra_loi
